# Funnel Drop-off Analysis by Step (Test vs Control)
## Why analyse both visits and clients?
We analysed drop-off at both the **visit level** and the **client level** to understand the funnel from two perspectives.

The visit-level analysis shows where visits ended, including cases where the same client made multiple attempts. This helps identify steps where clients may repeatedly leave the process.

The client-level analysis counts each client once, using their furthest recorded step across all visits. This shows how far clients progressed at least once during the observation period, even if they dropped off on an earlier visit.

Together, the two analyses show both what happened during visits and how far individual clients eventually progressed.

# Visit-Level Funnel Drop-off Analysis by Step (Test vs Control)

## Main KPI: Visit Drop-off Rate by Step

**Definition:** The share of visits whose furthest recorded funnel step was the current step, among visits whose furthest recorded step was the current step or later.

Each visit is counted once using its furthest recorded step. The rate is calculated for each transition (`start → step_1`, `step_1 → step_2`, `step_2 → step_3`, and `step_3 → confirm`), separately for Test and Control.

**Hypotheses: **

**H0:** The drop-off rate is the same in test and control (p_test = p_control).

**H1:** The drop-off rates differ (p_test ≠ p_control).

**Test:** Two-proportions z-test

**Significance level**: α = 0.05 / 4 = 0.0125

**Decision rule:** reject H0 if p-value < 0.0125.

In [301]:
import pandas as pd

In [302]:
df_events = pd.read_csv("../data/clean/df_events.csv")

In [303]:
df_events.head()

,client_id,visitor_id,visit_id,process_step,date_time,variation
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test


In [304]:
# To chek the unique values of steps
df_events["process_step"].unique().tolist()

['start', 'step_1', 'step_2', 'step_3', 'confirm']

In [305]:
stage_map = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

df_events["stage"] = df_events["process_step"].map(stage_map)

In [306]:
# Finding the furthest step of each event separately for test and control using the column stage, represents the step by number

furthest_step_test = df_events[df_events["variation"]=="Test"].groupby(["client_id","visitor_id","visit_id"])["stage"].max()

furthest_step_control = df_events[df_events["variation"]=="Control"].groupby(["client_id","visitor_id","visit_id"])["stage"].max()

In [307]:
furthest_step_test

client_id  visitor_id             visit_id                    
555        402506806_56087378777  637149525_38041617439_716659    4
647        66758770_53988066587   40369564_40101682850_311847     4
934        810392784_45004760546  7076463_57954418406_971348      0
1336       920624746_32603333901  583743392_96265099036_939815    4
                                  614001770_19101025926_112779    4
                                                                 ..
9999626    52633065_71189986073   182723364_27031318473_880901    1
9999729    604429154_69247391147  99583652_41711450505_426179     1
           834634258_21862004160  870243567_56915814033_814203    4
           843385170_36953471821  493310979_9209676464_421146     2
9999832    145538019_54444341400  472154369_16714624241_585315    1
Name: stage, Length: 37204, dtype: int64

In [308]:
furthest_step_control.unique()

array([3, 0, 2, 4, 1])

In [309]:
# A function to calculate the control drop-offs at each stage until step 3. Completion of 4 means a successful event.

def drop_count_control(k):
    drop_count = (furthest_step_control == k).sum()
    return drop_count
for k in (0,1,2,3):
    print(f"Drop count of control group at step {k} is {drop_count_control(k)}")
print(f"completed counts of control is {drop_count_control(4)}")

Drop count of control group at step 0 is 7572
Drop count of control group at step 1 is 3494
Drop count of control group at step 2 is 2040
Drop count of control group at step 3 is 3049
completed counts of control is 16088


In [310]:
# To check if all the rows are calculated

print(sum(drop_count_control(k) for k in range(5)), len(furthest_step_control))

32243 32243


In [311]:
# A function to calculate the test drop-offs at each stage until step 3. Completion of 4 means a successful event.

def drop_count_test(k):
    drop_count = (furthest_step_test == k).sum()
    return drop_count
for k in (0,1,2,3):
    print(f"Drop count of test group at step {k} is {drop_count_test(k)}")
print(f"completed counts of test is {drop_count_test(4)}")

Drop count of test group at step 0 is 5082
Drop count of test group at step 1 is 3908
Drop count of test group at step 2 is 2573
Drop count of test group at step 3 is 3850
completed counts of test is 21791


In [312]:
# To check if all the rows are calculated

print(sum(drop_count_test(k) for k in range(5)), len(furthest_step_test))

37204 37204


In [313]:
# Creating a data frames for test and control groups with the drop counts at each stage.

stage_names = {0: "start", 1: "step_1", 2: "step_2", 3: "step_3"}

drop_test_df = pd.DataFrame({
    "stage": [stage_names[k] for k in range(4)],
    "test_drops": [drop_count_test(k) for k in range(4)]
})

drop_control_df = pd.DataFrame({
    "stage": [stage_names[k] for k in range(4)],
    "control_drops": [drop_control(k) for k in range(4)]
})

In [314]:
drop_test_df

,stage,test_drops
0,start,5082
1,step_1,3908
2,step_2,2573
3,step_3,3850


In [315]:
drop_control_df

,stage,control_drops
0,start,7572
1,step_1,3494
2,step_2,2040
3,step_3,3049


In [316]:
# To test Two-proportions z-test

# n_t gives the total number of test counts that reached until that particular stage. 
# if stage =2, then n_t gives the total number of test counts that reached until step 2 or beyond.
n_t = sum(drop_count_test(j) for j in range(k, 5))

#x_t gives the count of drop_offs at each stage
x_t = drop_test_df["test_drops"][k]

# n_c gives the total number of control counts that reached until that particular stage. 
# if stage =1, then n_c gives the total number of control counts that reached until step 1 or beyond.
n_c = sum(drop_count_control(j) for j in range(k,5))

#x_c gives the control count of drop_offs at each stage
x_c = drop_control_df["control_drops"][k]

#p_t and p_c give proportion at each stage for test and control respectively. 
p_t = x_t / n_t
p_c = x_c / n_c

# Test propotion - control proportion
p_diff = (p_t) - (p_c)


In [317]:
# proportions_ztest

k = 0
results = []

while k < 4:
    n_t = sum(drop_count_test(j) for j in range(k, 5))
    x_t = drop_test_df["test_drops"][k]
    n_c = sum(drop_count_control(j) for j in range(k, 5))
    x_c = drop_control_df["control_drops"][k]

    p_t, p_c = x_t / n_t, x_c / n_c
    z, p_value = proportions_ztest([x_t, x_c], [n_t, n_c])

    results.append({"k": k, "n_t": n_t, "x_t": x_t, "n_c": n_c, "x_c": x_c,
                    "p_t": p_t, "p_c": p_c, "p_diff": p_t - p_c,
                    "z": z, "p_value": p_value})
    k += 1


results_df = pd.DataFrame(results)
results_df

,k,n_t,x_t,n_c,x_c,p_t,p_c,p_diff,z,p_value
0,0,37204,5082,32243,7572,0.136598,0.234842,-0.098243,-33.448881,2.671987e-245
1,1,32122,3908,24671,3494,0.121661,0.141624,-0.019963,-7.004236,2.483373e-12
2,2,28214,2573,21177,2040,0.091196,0.096331,-0.005135,-1.940932,5.226653e-02
3,3,25641,3850,19137,3049,0.150150,0.159325,-0.009175,-2.660340,7.806180e-03


In [318]:
alpha = 0.05 / 4
results_df["reject_H0"] = results_df["p_value"] < alpha
results_df["transition"] = ["start", "step_1", "step_2", "step_3"]
results_df

,k,n_t,x_t,n_c,x_c,p_t,p_c,p_diff,z,p_value,reject_H0,transition
0,0,37204,5082,32243,7572,0.136598,0.234842,-0.098243,-33.448881,2.671987e-245,True,start
1,1,32122,3908,24671,3494,0.121661,0.141624,-0.019963,-7.004236,2.483373e-12,True,step_1
2,2,28214,2573,21177,2040,0.091196,0.096331,-0.005135,-1.940932,5.226653e-02,False,step_2
3,3,25641,3850,19137,3049,0.150150,0.159325,-0.009175,-2.660340,7.806180e-03,True,step_3


In [319]:
final_table = pd.DataFrame({
    "transition": results_df["transition"],
    "drop_off_test_%": (results_df["p_t"] * 100).round(2),
    "drop_off_control_%": (results_df["p_c"] * 100).round(2),
    "pp(test-control)": (results_df["p_diff"] * 100).round(2),
    "z_statistic": results_df["z"].round(2),
    "p_value": results_df["p_value"],
    "significant": results_df["reject_H0"].map({True: "Yes", False: "No"})
})
final_table

,transition,drop_off_test_%,drop_off_control_%,pp(test-control),z_statistic,p_value,significant
0,start,13.66,23.48,-9.82,-33.45,2.671987e-245,Yes
1,step_1,12.17,14.16,-2.00,-7.00,2.483373e-12,Yes
2,step_2,9.12,9.63,-0.51,-1.94,5.226653e-02,No
3,step_3,15.02,15.93,-0.92,-2.66,7.806180e-03,Yes


In [320]:
def show_statistical_test(statistic: float, alpha: float, n: int,
                          distribution: str = "normal", alternative: str = "two-sided"):
    if distribution not in ["t-student", "normal"]:
        raise TypeError("Only 't-student' and 'normal' are accepted")
    if alternative not in ["two-sided", "lower", "greater"]:
        raise TypeError("Only 'two-sided', 'lower' and 'greater' are accepted")
    if not isinstance(statistic, float) or not isinstance(alpha, float) or not isinstance(n, int):
        raise TypeError("statistic and alpha must be float, n must be int")

    d = st.norm() if distribution == "normal" else st.t(df=n - 1)
    x = np.linspace(-3, 3)
    fig = px.line(pd.DataFrame({"x": x, "pdf": d.pdf(x)}), x="x", y="pdf")

    def tail(a, b):  # red rejection area
        xs = np.linspace(a, b)
        fig.add_scatter(x=xs, y=d.pdf(xs), fill="tozeroy", mode="none", fillcolor="red")

    def label(x_pos, text, anchor):  # critical value line + label
        fig.add_vline(x=x_pos, line_color="red")
        fig.add_annotation(x=x_pos, y=0.08, text=text, xref="x", yref="paper",
                           xanchor=anchor, showarrow=False)

    if alternative == "two-sided":
        lo, hi = d.ppf(alpha / 2), d.ppf(1 - alpha / 2)
        tail(-3, lo); tail(hi, 3)
        label(lo, f"Lower critical value {lo:.2f}", "right")
        label(hi, f"Upper critical value {hi:.2f}", "left")
    elif alternative == "lower":
        c = d.ppf(alpha)
        tail(-3, c)
        label(c, f"Critical value {c:.2f}", "right")
    else:
        c = d.ppf(1 - alpha)
        tail(c, 3)
        label(c, f"Critical value {c:.2f}", "left")

    fig.add_vline(x=statistic)
    fig.add_annotation(x=statistic, y=1, text=f"Statistic {statistic:.2f}",
                       xref="x", yref="paper", yanchor="top", showarrow=False)

    fig.update_layout(title_text=f"{distribution} Probability Density Function",
                  title_x=0.5, showlegend=False,
                  height=380, width=750, margin=dict(t=50, b=40, l=40, r=20))
    fig.update_xaxes(range=[-max(4, abs(statistic) + 1), max(4, abs(statistic) + 1)])
    fig.show()

In [321]:
for _, row in results_df.iterrows():
    print(row["transition"])
    show_statistical_test(float(row["z"]), float(alpha), int(row["n_t"] + row["n_c"]),
                          distribution="normal", alternative="two-sided")

start


step_1


step_2


step_3


Visit-level analysis: At α = 0.0125, we reject H₀ for start, step_1, and step_3 because their p-values are below 0.0125. We do not reject H₀ for step_2 because its p-value is above 0.0125. These visit-level p-values should be interpreted cautiously because clients can have multiple visits

In [322]:
final_table.to_csv("../data/clean/dropoff_results_event.csv", index=False)

# Client-Level Funnel Drop-off Analysis by Step (Test vs Control)

## Main KPI: Client Drop-off Rate by Step

**Definition:** The share of clients whose furthest recorded funnel step was the current step, among clients whose furthest recorded step was the current step or later.

Each client is counted once using their furthest recorded step across all visits. The rate is calculated for each transition (`start → step_1`, `step_1 → step_2`, `step_2 → step_3`, and `step_3 → confirm`), separately for Test and Control.

**Hypotheses:**

**H₀:** The client drop-off rate is the same in Test and Control (p_test = p_control).

**H₁:** The client drop-off rates differ between Test and Control (p_test ≠ p_control).

**Test:** Two-proportions z-test, using client-level counts.

**Significance level:** α = 0.05 / 4 = 0.0125, using the Bonferroni correction for four transitions.

**Decision rule:** Reject H₀ for a transition if its p-value < 0.0125.

In [323]:
df_events.head()

,client_id,visitor_id,visit_id,process_step,date_time,variation,stage
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,1
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,3
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,4


In [324]:
furthest_client_test = (df_events[df_events["variation"] == "Test"].groupby("client_id")["stage"].max())

furthest_client_control = (df_events[df_events["variation"] == "Control"].groupby("client_id")["stage"].max())

In [325]:
def client_drop_count_control(k):
    drop_count = (furthest_client_control == k).sum()
    return drop_count

for k in (0, 1, 2, 3):
    print(f"Client drop count of control group at step {k} is {client_drop_count_control(k)}")

print(f"Completed client count of control is {client_drop_count_control(4)}")

Client drop count of control group at step 0 is 3285
Client drop count of control group at step 1 is 1455
Client drop count of control group at step 2 is 1265
Client drop count of control group at step 3 is 2093
Completed client count of control is 15434


In [326]:
def client_drop_count_test(k):
    drop_count = (furthest_client_test == k).sum()
    return drop_count

for k in (0, 1, 2, 3):
    print(f"Client drop count of test group at step {k} is {client_drop_count_test(k)}")

print(f"Completed client count of test is {client_drop_count_test(4)}")

Client drop count of test group at step 0 is 2454
Client drop count of test group at step 1 is 1980
Client drop count of test group at step 2 is 1411
Client drop count of test group at step 3 is 2436
Completed client count of test is 18687


In [327]:
print(sum(client_drop_count_control(k) for k in range(5)), len(furthest_client_control))
print(sum(client_drop_count_test(k) for k in range(5)), len(furthest_client_test))

23532 23532
26968 26968


In [328]:
stage_names = {0: "start", 1: "step_1", 2: "step_2", 3: "step_3"}

client_drop_test_df = pd.DataFrame({
    "stage": [stage_names[k] for k in range(4)],
    "test_drops": [client_drop_count_test(k) for k in range(4)]
})

client_drop_control_df = pd.DataFrame({
    "stage": [stage_names[k] for k in range(4)],
    "control_drops": [client_drop_count_control(k) for k in range(4)]
})

In [329]:
client_drop_test_df

,stage,test_drops
0,start,2454
1,step_1,1980
2,step_2,1411
3,step_3,2436


In [330]:
client_drop_control_df

,stage,control_drops
0,start,3285
1,step_1,1455
2,step_2,1265
3,step_3,2093


In [331]:
k = 0
client_results = []

while k < 4:
    n_t = sum(client_drop_count_test(j) for j in range(k, 5))
    x_t = client_drop_test_df["test_drops"][k]

    n_c = sum(client_drop_count_control(j) for j in range(k, 5))
    x_c = client_drop_control_df["control_drops"][k]

    p_t, p_c = x_t / n_t, x_c / n_c
    z, p_value = proportions_ztest([x_t, x_c], [n_t, n_c])

    client_results.append({
        "k": k,
        "n_t": n_t,
        "x_t": x_t,
        "n_c": n_c,
        "x_c": x_c,
        "p_t": p_t,
        "p_c": p_c,
        "p_diff": p_t - p_c,
        "z": z,
        "p_value": p_value
    })

    k += 1

client_results_df = pd.DataFrame(client_results)
client_results_df

,k,n_t,x_t,n_c,x_c,p_t,p_c,p_diff,z,p_value
0,0,26968,2454,23532,3285,0.090997,0.139597,-0.048600,-17.166090,4.764535e-66
1,1,24514,1980,20247,1455,0.080770,0.071862,0.008908,3.523924,4.252056e-04
2,2,22534,1411,18792,1265,0.062616,0.067316,-0.004699,-1.933040,5.323125e-02
3,3,21123,2436,17527,2093,0.115325,0.119416,-0.004091,-1.244940,2.131540e-01


In [332]:
alpha = 0.05 / 4

client_results_df["reject_H0"] = client_results_df["p_value"] < alpha
client_results_df["transition"] = ["start", "step_1", "step_2", "step_3"]

client_results_df

,k,n_t,x_t,n_c,x_c,p_t,p_c,p_diff,z,p_value,reject_H0,transition
0,0,26968,2454,23532,3285,0.090997,0.139597,-0.048600,-17.166090,4.764535e-66,True,start
1,1,24514,1980,20247,1455,0.080770,0.071862,0.008908,3.523924,4.252056e-04,True,step_1
2,2,22534,1411,18792,1265,0.062616,0.067316,-0.004699,-1.933040,5.323125e-02,False,step_2
3,3,21123,2436,17527,2093,0.115325,0.119416,-0.004091,-1.244940,2.131540e-01,False,step_3


In [333]:
client_final_table = pd.DataFrame({
    "transition": client_results_df["transition"],
    "drop_off_test_%": (client_results_df["p_t"] * 100).round(2),
    "drop_off_control_%": (client_results_df["p_c"] * 100).round(2),
    "pp(test-control)": (client_results_df["p_diff"] * 100).round(2),
    "z_statistic": client_results_df["z"].round(2),
    "p_value": client_results_df["p_value"],
    "significant": client_results_df["reject_H0"].map({True: "Yes", False: "No"})
})

client_final_table

,transition,drop_off_test_%,drop_off_control_%,pp(test-control),z_statistic,p_value,significant
0,start,9.10,13.96,-4.86,-17.17,4.764535e-66,Yes
1,step_1,8.08,7.19,0.89,3.52,4.252056e-04,Yes
2,step_2,6.26,6.73,-0.47,-1.93,5.323125e-02,No
3,step_3,11.53,11.94,-0.41,-1.24,2.131540e-01,No


In [334]:
for _, row in client_results_df.iterrows():
    print(row["transition"])
    show_statistical_test(
        float(row["z"]),
        float(alpha),
        int(row["n_t"] + row["n_c"]),
        distribution="normal",
        alternative="two-sided"
    )

start


step_1


step_2


step_3


Visit-level analysis: At α = 0.0125, we reject H₀ for start and step_1 because their p-values are below 0.0125. Drop-off was lower in Test at start, but higher in Test at step_1.
We do not reject H₀ for step_2 and step_3 because their p-values are above 0.0125. There is insufficient evidence of a difference between Test and Control at these steps.

In [335]:
client_final_table.to_csv(
    "../data/clean/dropoff_results_client.csv",
    index=False
)

## Main KPI: Average Calls per Client

**Definition:** The mean number of calls made by clients during the recorded six-month period, calculated separately for Test and Control.

**Hypotheses:**

**H0:** The mean number of calls per client is the same in Test and Control.

**H1:** The mean number of calls per client differs between Test and Control.

**Test:** Two-sided Welch’s t-test.

**Significance level:** α = 0.05.

**Decision rule:** Reject H₀ if the p-value is below 0.05; otherwise, do not reject H₀.

In [336]:
import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import plotly.express as px

In [337]:
df_demo = pd.read_csv("../data/clean/df_demo_clean.csv")
df_roster = pd.read_csv ("../data/clean/roster_clean.csv")


In [338]:
df_clients = pd.merge(df_demo,df_roster,on="client_id",how="inner")
df_clients.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,variation
0,836976,6,73,60.5,U,2,45105.30,6,9,Test
1,2304905,7,94,58.0,U,2,110860.30,6,9,Control
2,1439522,5,64,32.0,U,2,52467.79,6,9,Test
3,1562045,16,198,49.0,M,2,67454.65,3,6,Test
4,5126305,12,145,33.0,F,2,103671.75,0,3,Control


In [339]:
df_clients["calls_6_mnth"].value_counts()

calls_6_mnth
6    12225
1     7908
0     7732
2     7215
3     5853
4     4848
5     4707
Name: count, dtype: int64

In [340]:
#calculating the mean, median, count of control and test 
calls_summary = (df_clients.groupby("variation")["calls_6_mnth"].agg(["count", "mean", "median"]).round(2))
calls_summary

,count,mean,median
variation,,,
Control,23527,3.13,3.0
Test,26961,3.06,3.0


In [341]:
# Creating individual columns with calls data
test_calls = df_clients[df_clients["variation"] == "Test"]["calls_6_mnth"]

control_calls = df_clients[df_clients["variation"] == "Control"]["calls_6_mnth"]


In [342]:
# Using Welch's t-test

t_stat, p_value = st.ttest_ind(test_calls,control_calls,equal_var=False)

In [343]:
# creating the final result table

alpha_calls = 0.05

calls_results = pd.DataFrame({
    "test_mean": [test_calls.mean()],
    "control_mean": [control_calls.mean()],
    "difference(t-c)": [test_calls.mean() - control_calls.mean()],
    "t_statistic": [t_stat],
    "p_value": [p_value],
    "reject_H0": [p_value < alpha_calls]
})

calls_results.round(4)

,test_mean,control_mean,difference(t-c),t_statistic,p_value,reject_H0
0,3.0619,3.1292,-0.0673,-3.4473,0.0006,True


In [344]:
def show_statistical_test(statistic: float, alpha: float, n: int,
                          distribution: str = "normal", alternative: str = "two-sided",df_welch= None):
    if distribution not in ["t-student", "normal"]:
        raise TypeError("Only 't-student' and 'normal' are accepted")
    if alternative not in ["two-sided", "lower", "greater"]:
        raise TypeError("Only 'two-sided', 'lower' and 'greater' are accepted")
    if not isinstance(statistic, float) or not isinstance(alpha, float) or not isinstance(n, int):
        raise TypeError("statistic and alpha must be float, n must be int")

    d = st.norm() if distribution == "normal" else st.t(df=df_welch if df_welch is not None else n - 1)
    x = np.linspace(-3, 3)
    fig = px.line(pd.DataFrame({"x": x, "pdf": d.pdf(x)}), x="x", y="pdf")

    def tail(a, b):  # red rejection area
        xs = np.linspace(a, b)
        fig.add_scatter(x=xs, y=d.pdf(xs), fill="tozeroy", mode="none", fillcolor="red")

    def label(x_pos, text, anchor):  # critical value line + label
        fig.add_vline(x=x_pos, line_color="red")
        fig.add_annotation(x=x_pos, y=0.08, text=text, xref="x", yref="paper",
                           xanchor=anchor, showarrow=False)

    if alternative == "two-sided":
        lo, hi = d.ppf(alpha / 2), d.ppf(1 - alpha / 2)
        tail(-3, lo); tail(hi, 3)
        label(lo, f"Lower critical value {lo:.2f}", "right")
        label(hi, f"Upper critical value {hi:.2f}", "left")
    elif alternative == "lower":
        c = d.ppf(alpha)
        tail(-3, c)
        label(c, f"Critical value {c:.2f}", "right")
    else:
        c = d.ppf(1 - alpha)
        tail(c, 3)
        label(c, f"Critical value {c:.2f}", "left")

    fig.add_vline(x=statistic)
    fig.add_annotation(x=statistic, y=1, text=f"Statistic {statistic:.2f}",
                       xref="x", yref="paper", yanchor="top", showarrow=False)

    fig.update_layout(title_text=f"{distribution} Probability Density Function",
                  title_x=0.5, showlegend=False,
                  height=380, width=750, margin=dict(t=50, b=40, l=40, r=20))
    fig.update_xaxes(range=[-max(4, abs(statistic) + 1), max(4, abs(statistic) + 1)])
    fig.show()

In [345]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [347]:
n_t = len(test_calls)
n_c = len(control_calls)

var_t = test_calls.var()
var_c = control_calls.var()

df_welch = (
    (var_t / n_t + var_c / n_c) ** 2
    / (
        (var_t / n_t) ** 2 / (n_t - 1)
        + (var_c / n_c) ** 2 / (n_c - 1)
    )
)

In [348]:
show_statistical_test(
    float(t_stat),
    float(alpha_calls),
    int(len(test_calls) + len(control_calls)),
    distribution="t-student",
    alternative="two-sided",
    df_welch=df_welch
)

At **α = 0.05**, we reject H₀ because the p-value (**0.0006**) is below 0.05. Test clients averaged **3.0619 calls**, compared with **3.1292 calls** for Control clients—a statistically significant difference of **0.0673 fewer calls per client** in Test.

The difference is small, and the result alone does not establish that the new design caused it.

## KPI: Completion Rate by Call Status

**Definition:** The percentage of clients who reached `confirm` at least once, compared between clients with at least one recorded call and clients with zero recorded calls. Calculate the rates separately for Test and Control.

## Hypotheses

For each variation separately:

**H0:** Clients with calls have an equal or lower completion rate than clients with zero calls.

**H1:** Clients with calls have a higher completion rate than clients with zero calls.

**Test:** One-sided two-proportions z-test.

**Significance level:** α = 0.025 per test.


In [ ]:
df_clients["calls_6_mnth"].unique()

In [ ]:
df_clients[df_clients["calls_6_mnth"] == 0]["variation"].value_counts()

In [ ]:
furthest_client = pd.concat([furthest_client_test,furthest_client_control]).reset_index(name = "furthest_step")

clients = pd.merge(df_clients,furthest_client,on = "client_id",how = "inner")

clients

In [ ]:
clients["calls_6_mnth"].value_counts().sort_index()

In [ ]:

# Test clients who did not make calls and completed the event

test_clients_completion_zero_calls = clients[
    (clients["calls_6_mnth"] == 0) & 
    (clients["furthest_step"] == 4) & 
    (clients["variation"] == "Test")
]

# Test clients who made calls

test_clients_completion_with_calls = clients[
    (clients["calls_6_mnth"] > 0) & 
    (clients["furthest_step"] == 4) & 
    (clients["variation"] == "Test")
]

# Total test clients with zero calls

total_test_clients_zero_calls = clients[
    (clients["calls_6_mnth"] == 0) & 
    (clients["variation"] == "Test")
]

# Total test clients with at least one call

total_test_clients_with_calls = clients[
    (clients["calls_6_mnth"] > 0) & 
    (clients["variation"] == "Test")
]

# Percentage of test clients who completed the event and did not make calls

test_completed_zero_calls= round((len(test_clients_completion_zero_calls) / len(total_test_clients_zero_calls)) * 100,2)

# Percentage of test clients who completed the event and made at least one call

test_completed_with_calls = round((len(test_clients_completion_with_calls) / len(total_test_clients_with_calls)) * 100,2)


In [ ]:
# Percentage of test clients who completed the event and did not make calls
test_completed_zero_calls

In [ ]:
# Percentage of test clients who completed the event and made at least one call
test_completed_with_calls

In [ ]:
# control clients who did not make calls and completed the event

control_clients_completion_zero_calls = clients[
    (clients["calls_6_mnth"] == 0) & 
    (clients["furthest_step"] == 4) & 
    (clients["variation"] == "Control")
]

# control clients who made calls

control_clients_completion_with_calls = clients[
    (clients["calls_6_mnth"] > 0) & 
    (clients["furthest_step"] == 4) & 
    (clients["variation"] == "Control")
]

# Total control clients with zero calls

total_control_clients_zero_calls = clients[
    (clients["calls_6_mnth"] == 0) & 
    (clients["variation"] == "Control")
]

# Total control clients with at least one call

total_control_clients_with_calls = clients[
    (clients["calls_6_mnth"] > 0) & 
    (clients["variation"] == "Control")
]

# Percentage of control clients who completed the event and did not make calls

control_completed_zero_calls= round((len(control_clients_completion_zero_calls) / len(total_control_clients_zero_calls)) * 100,2)

# Percentage of control clients who completed the event and made at least one call

control_completed_with_calls = round((len(control_clients_completion_with_calls) / len(total_control_clients_with_calls)) * 100,2)


In [351]:
# Percentage of control clients who completed the event and did not make calls

control_completed_zero_calls

62.79

In [352]:
# Percentage of control clients who completed the event and made at least one call

control_completed_with_calls

66.07

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [353]:
z_test, p_test_higher = proportions_ztest(
    [len(test_clients_completion_with_calls),
     len(test_clients_completion_zero_calls)],
    [len(total_test_clients_with_calls),
     len(total_test_clients_zero_calls)],
    alternative="larger"
)

z_control, p_control_higher = proportions_ztest(
    [len(control_clients_completion_with_calls),
     len(control_clients_completion_zero_calls)],
    [len(total_control_clients_with_calls),
     len(total_control_clients_zero_calls)],
    alternative="larger"
)

print("Test:", z_test, p_test_higher)
print("Control:", z_control, p_control_higher)

Test: 5.863841088627766 2.261401770650671e-09
Control: 3.759727746851684 8.50491833353464e-05


In [354]:
def show_statistical_test(statistic: float, alpha: float, n: int,
                          distribution: str = "normal", alternative: str = "lower"):
    if distribution not in ["t-student", "normal"]:
        raise TypeError("Only 't-student' and 'normal' are accepted")
    if alternative not in ["two-sided", "lower", "greater"]:
        raise TypeError("Only 'two-sided', 'lower' and 'greater' are accepted")
    if not isinstance(statistic, float) or not isinstance(alpha, float) or not isinstance(n, int):
        raise TypeError("statistic and alpha must be float, n must be int")

    d = st.norm() if distribution == "normal" else st.t(df=n - 1)
    x = np.linspace(-3, 3)
    fig = px.line(pd.DataFrame({"x": x, "pdf": d.pdf(x)}), x="x", y="pdf")

    def tail(a, b):  # red rejection area
        xs = np.linspace(a, b)
        fig.add_scatter(x=xs, y=d.pdf(xs), fill="tozeroy", mode="none", fillcolor="red")

    def label(x_pos, text, anchor):  # critical value line + label
        fig.add_vline(x=x_pos, line_color="red")
        fig.add_annotation(x=x_pos, y=0.08, text=text, xref="x", yref="paper",
                           xanchor=anchor, showarrow=False)

    if alternative == "two-sided":
        lo, hi = d.ppf(alpha / 2), d.ppf(1 - alpha / 2)
        tail(-3, lo); tail(hi, 3)
        label(lo, f"Lower critical value {lo:.2f}", "right")
        label(hi, f"Upper critical value {hi:.2f}", "left")
    elif alternative == "lower":
        c = d.ppf(alpha)
        tail(-3, c)
        label(c, f"Critical value {c:.2f}", "right")
    else:
        c = d.ppf(1 - alpha)
        tail(c, 3)
        label(c, f"Critical value {c:.2f}", "left")

    fig.add_vline(x=statistic)
    fig.add_annotation(x=statistic, y=1, text=f"Statistic {statistic:.2f}",
                       xref="x", yref="paper", yanchor="top", showarrow=False)

    fig.update_layout(title_text=f"{distribution} Probability Density Function",
                  title_x=0.5, showlegend=False,
                  height=380, width=750, margin=dict(t=50, b=40, l=40, r=20))
    fig.update_xaxes(range=[-max(4, abs(statistic) + 1), max(4, abs(statistic) + 1)])
    fig.show()

In [358]:
# Test group
show_statistical_test(
    float(z_test),
    float(0.025),
    int(len(total_test_clients_with_calls) + len(total_test_clients_zero_calls)),
    distribution="normal",
    alternative="greater"
)

In [357]:
# Control group
show_statistical_test(
    float(z_control),
    float(0.025),
    int(len(total_control_clients_with_calls) + len(total_control_clients_zero_calls)),
    distribution="normal",
    alternative="greater"
)

**Insight:** At α = 0.025, we reject H₀ for both Test and Control. Clients with at least one recorded call had a statistically higher completion rate than clients with zero calls.

Test: 70.00% with calls versus 65.48% with zero calls.
Control: 66.07% with calls versus 62.79% with zero calls.
